# 07. cuDF Import/Export and GPU Acceleration
## 📚 Learning Objectives

By completing this notebook, you will:
- Import and export data in different formats using cuDF functions
- Use GPU acceleration from cuDF to process data faster
- Compare cuDF performance with Pandas
- Optimize data processing using GPU acceleration

## 🔗 Where this fits

**Builds on:** Course 05 — Unit 1, lesson 03 "Introduction to cuDF" — the API tour, now applied to the loading and cleaning work of this unit.

**Used later in:** Course 05 — Unit 5, lesson 04, which places cuDF inside a full RAPIDS workflow.

---

This notebook covers practical activities from **Course 05, Unit 2**:
- Import/Export using cuDF: Importing and exporting data in different formats using cuDF functions
- Optimization using cuDF: Using GPU acceleration from cuDF to process data faster

---

## Introduction to cuDF

**cuDF** is a GPU-accelerated DataFrame library that provides a pandas-like API for working with data on GPUs. It's part of the RAPIDS ecosystem and can significantly speed up data processing operations.


## 🎯 The case: why Twitter and Cloudera built a new file format

By 2012 both Twitter and Cloudera were storing petabytes of data as text and row-oriented
files, and paying for it twice — once in disk, once in the CPU time spent re-parsing every
number from characters on every single query. Their answer, built jointly and released in
**July 2013**, was **Apache Parquet**: store each *column* contiguously, with its type
declared once and compression chosen per column, so a query that needs 3 of 79 columns
reads 3 of 79 columns. Parquet became a top-level Apache project on **27 April 2015** and
is now the default storage format for essentially every analytics engine — Spark, Dask,
DuckDB, BigQuery, and cuDF.

**What goes wrong without this.** CSV re-encodes every float as text. Reading it means
parsing every character of every number, every time, on one CPU core — and it carries no
types, so `read_csv` has to *guess* (the failure that corrupted 19.6% of the genomics
literature, lesson 01). You will measure the cost below on a real 284,807-row transaction
file: **CSV 20.0 MB against Parquet 9.3 MB**, and a read of **0.077 s against 0.026 s**.

The notebook is careful about the second number, and you should be too: size is a
reliable win, speed is workload-dependent. Re-run the cell and the timings will move.


## The Story

**BEFORE**: You know pandas but don't know how to accelerate data processing with GPU.

**AFTER**: You'll learn cuDF - GPU-accelerated data processing for large datasets (vendors report order-of-magnitude speedups at scale; this notebook measures real timings only when a GPU is present)!

**Why this matters**: cuDF Import/Export and GPU Acceleration is essential for building complete, professional data science solutions!

---

🚀 Google Colab Setup (Run this first if using Colab)


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- CSV/Parquet files
- cuDF (or pandas fallback)

**Outputs:** What you'll see when you run the cells

- GPU load/save timings
- DataFrames

---

In [1]:
# WHAT: Try to import cuDF; fall back to pandas-only mode when no GPU stack exists.
# WHY: The guard keeps the notebook honest and runnable anywhere - GPU code only runs when a GPU is really present.

# Try importing cuDF (requires CUDA and RAPIDS installation)
try:
    import cudf
    import numpy as np
    import pandas as pd
    CUDF_AVAILABLE = True
    print("✅ cuDF imported successfully!")
    print(f"cuDF version: {cudf.__version__}")
except ImportError:
    CUDF_AVAILABLE = False
    import numpy as np
    import pandas as pd
    print("⚠️  cuDF not available. Install RAPIDS for GPU acceleration:")
    print("   Note: Requires CUDA-capable GPU and RAPIDS installation")
    print("   Continuing with Pandas examples...")

print("✅ Libraries imported!")

⚠️  cuDF not available. Install RAPIDS for GPU acceleration:
   Note: Requires CUDA-capable GPU and RAPIDS installation
   Continuing with Pandas examples...
✅ Libraries imported!


## Part 1: Data Import with cuDF

cuDF supports importing data from CSV, Parquet, JSON, and other formats, similar to
Pandas but with GPU acceleration.

We import a real file: `creditcard_fraud.csv` — 284,807 anonymised card
transactions, of which 492 are fraud. Big enough that format and engine choices
actually matter.


In [2]:
# WHAT: Read a real 151 MB transaction file with pandas (and with cuDF if a GPU is present).
# WHY: Import performance only means something on a file large enough to feel it - so we use a real one, not a generated toy.

DATA_DIR = '../../../Course 04/datasets/raw/'
SOURCE_CSV = DATA_DIR + 'creditcard_fraud.csv'

# usecols keeps this affordable: 6 of the 31 columns is all these examples need.
COLS = ['Time', 'V1', 'V2', 'V14', 'Amount', 'Class']

if CUDF_AVAILABLE:
    # Import with cuDF (GPU-accelerated)
    print("\n" + "=" * 60)
    print("Importing with cuDF (GPU-accelerated):")
    print("=" * 60)
    df_cudf = cudf.read_csv(SOURCE_CSV, usecols=COLS)
    print(f"DataFrame shape: {df_cudf.shape}")
    print(f"\nFirst few rows:")
    print(df_cudf.head())

    # Import with Pandas (CPU)
    print("\n" + "=" * 60)
    print("Importing with Pandas (CPU):")
    print("=" * 60)
    df_pandas = pd.read_csv(SOURCE_CSV, usecols=COLS)
    print(f"DataFrame shape: {df_pandas.shape}")
else:
    # No GPU here: measure the CPU path only, and say so.
    print("\n" + "=" * 60)
    print("Importing with Pandas (CPU):")
    print("=" * 60)
    df_pandas = pd.read_csv(SOURCE_CSV, usecols=COLS)
    print(f"DataFrame shape: {df_pandas.shape}")
    print(f"\nFirst few rows:")
    print(df_pandas.head())

print(f"\nReal class balance in this file:"
      f" {int(df_pandas['Class'].sum())} fraudulent out of {len(df_pandas):,}"
      f" ({df_pandas['Class'].mean() * 100:.3f}%)")
print("That extreme imbalance is a property of the real world, and it will")
print("dominate every modelling decision in Unit 4.")



Importing with Pandas (CPU):


DataFrame shape: (284807, 6)

First few rows:
   Time        V1        V2       V14  Amount  Class
0   0.0 -1.359807 -0.072781 -0.311169  149.62      0
1   0.0  1.191857  0.266151 -0.143772    2.69      0
2   1.0 -1.358354 -1.340163 -0.165946  378.66      0
3   1.0 -0.966272 -0.185226 -0.287924  123.50      0
4   2.0 -1.158233  0.877737 -1.119670   69.99      0

Real class balance in this file: 492 fraudulent out of 284,807 (0.173%)
That extreme imbalance is a property of the real world, and it will
dominate every modelling decision in Unit 4.


## Part 2: Data Export with cuDF

Exporting data to different formats (CSV, Parquet, JSON) using cuDF.


In [3]:
# WHAT: Export the real DataFrame to CSV and Parquet and compare the resulting file sizes and read times.
# WHY: Format choice is a measurable trade-off, not a slogan - so we measure size and read time instead of quoting folklore.

import os
import time

print("=" * 60)
print("Exporting and re-reading: CSV vs Parquet (measured)")
print("=" * 60)

if CUDF_AVAILABLE:
    df_cudf.to_csv('output_cudf.csv', index=False)
    df_cudf.to_parquet('output_cudf.parquet')
    print("✅ Exported with cuDF: output_cudf.csv, output_cudf.parquet")
    csv_path, parquet_path = 'output_cudf.csv', 'output_cudf.parquet'
else:
    df_pandas.to_csv('output_pandas.csv', index=False)
    df_pandas.to_parquet('output_pandas.parquet')
    print("✅ Exported with pandas: output_pandas.csv, output_pandas.parquet")
    print("   (cuDF would use the identical .to_csv/.to_parquet calls on a GPU)")
    csv_path, parquet_path = 'output_pandas.csv', 'output_pandas.parquet'

# Measure, do not assert: how much smaller and faster is Parquet on THIS data?
csv_mb = os.path.getsize(csv_path) / 1e6
parquet_mb = os.path.getsize(parquet_path) / 1e6

t0 = time.time(); _ = pd.read_csv(csv_path); csv_read = time.time() - t0
t0 = time.time(); _ = pd.read_parquet(parquet_path); parquet_read = time.time() - t0

print(f"\nFile size   CSV: {csv_mb:6.1f} MB   Parquet: {parquet_mb:6.1f} MB"
      f"   -> Parquet is {csv_mb / parquet_mb:.1f}x smaller")
print(f"Read time   CSV: {csv_read:6.3f} s    Parquet: {parquet_read:6.3f} s", end="")
if parquet_read < csv_read:
    print(f"   -> Parquet is {csv_read / parquet_read:.1f}x faster")
else:
    print(f"   -> CSV was {parquet_read / csv_read:.1f}x FASTER here")

print("\nParquet reliably wins on SIZE: it stores columns with declared types and")
print("compression instead of re-encoding every number as text.")
print("Read SPEED is a separate question and the answer is not fixed. Parquet skips")
print("parsing entirely, which usually wins; but it also pays per-file metadata and")
print("decompression cost, so on a small, narrow, warm-cached table the difference")
print(f"can shrink or even reverse. This run measured {len(COLS)} columns x"
      f" {len(df_pandas):,} rows.")
print("Lesson: the numbers above came from YOUR machine on THIS access pattern.")
print("Re-run the cell and they will move. A format is not faster in the abstract -")
print("benchmark the shape of data you actually have.")
print("\nNote: cuDF supports CSV, Parquet, JSON, ORC and other formats with the")
print("same method names, so this code moves to a GPU unchanged.")


Exporting and re-reading: CSV vs Parquet (measured)


✅ Exported with pandas: output_pandas.csv, output_pandas.parquet
   (cuDF would use the identical .to_csv/.to_parquet calls on a GPU)

File size   CSV:   20.0 MB   Parquet:    9.3 MB   -> Parquet is 2.1x smaller
Read time   CSV:  0.077 s    Parquet:  0.026 s   -> Parquet is 3.0x faster

Parquet reliably wins on SIZE: it stores columns with declared types and
compression instead of re-encoding every number as text.
Read SPEED is a separate question and the answer is not fixed. Parquet skips
parsing entirely, which usually wins; but it also pays per-file metadata and
decompression cost, so on a small, narrow, warm-cached table the difference
can shrink or even reverse. This run measured 6 columns x 284,807 rows.
Lesson: the numbers above came from YOUR machine on THIS access pattern.
Re-run the cell and they will move. A format is not faster in the abstract -
benchmark the shape of data you actually have.

Note: cuDF supports CSV, Parquet, JSON, ORC and other formats with the
same method

## Part 3: GPU Acceleration Performance Comparison

Let's compare cuDF (GPU) vs Pandas (CPU) performance for common operations.

**Honesty note**: on a machine without a CUDA GPU, cuDF cannot run — in that
case the cell below measures the *pandas CPU baseline only* and clearly says
that no GPU numbers were produced here.


In [4]:
# WHAT: Time filter/groupby operations on the real 284,807-row transaction table (GPU vs CPU when cuDF exists).
# WHY: Measured timings - not vendor claims - show whether the GPU pays off at this data size.

import time

# The real table loaded above IS the benchmark data - no generated stand-in.
large_data = df_pandas
print(f"Benchmark data: {len(large_data):,} real transactions x"
      f" {large_data.shape[1]} columns")

# A meaningful grouping key: bucket transactions by amount, which is what an
# anti-fraud analyst would actually aggregate on.
large_data = large_data.assign(
    amount_band=pd.cut(large_data['Amount'],
                       bins=[-0.01, 10, 50, 200, 1000, 30000],
                       labels=['0-10', '10-50', '50-200', '200-1k', '1k+']))

if CUDF_AVAILABLE:
    large_cudf = cudf.from_pandas(large_data)

    print("=" * 60)
    print("Performance Comparison: cuDF (GPU) vs Pandas (CPU)")
    print("=" * 60)

    print("\n1. Filtering operations:")
    start = time.time()
    filtered_cudf = large_cudf[large_cudf['Amount'] > 200]
    cudf_time = time.time() - start
    print(f"   cuDF (GPU): {cudf_time:.4f} seconds")

    start = time.time()
    filtered_pandas = large_data[large_data['Amount'] > 200]
    pandas_time = time.time() - start
    print(f"   Pandas (CPU): {pandas_time:.4f} seconds")
    print(f"   Speedup: {pandas_time / cudf_time:.2f}x faster with cuDF")

    print("\n2. Groupby operations:")
    start = time.time()
    grouped_cudf = large_cudf.groupby('amount_band')['Class'].mean()
    cudf_time = time.time() - start
    print(f"   cuDF (GPU): {cudf_time:.4f} seconds")

    start = time.time()
    grouped_pandas = large_data.groupby('amount_band', observed=True)['Class'].mean()
    pandas_time = time.time() - start
    print(f"   Pandas (CPU): {pandas_time:.4f} seconds")
    print(f"   Speedup: {pandas_time / cudf_time:.2f}x faster with cuDF")
else:
    print("=" * 60)
    print("No CUDA GPU here - measuring the pandas (CPU) baseline only")
    print("=" * 60)

    start = time.time()
    filtered_pandas = large_data[large_data['Amount'] > 200]
    pandas_filter_time = time.time() - start
    print(f"\n1. Filtering {len(large_data):,} rows:")
    print(f"   Pandas (CPU): {pandas_filter_time:.4f} seconds"
          f" → {len(filtered_pandas):,} rows kept")

    start = time.time()
    grouped_pandas = large_data.groupby('amount_band', observed=True)['Class'].mean()
    pandas_group_time = time.time() - start
    print(f"\n2. Groupby-mean over"
          f" {large_data['amount_band'].nunique()} amount bands:")
    print(f"   Pandas (CPU): {pandas_group_time:.4f} seconds")

    print("""
What cuDF would add (NOT measured here - no GPU ran in this notebook):
    - The same filter/groupby API, executed on the GPU
    - NVIDIA's published benchmarks report large speedups on much bigger
      datasets - treat those as the vendor's numbers, not ours
    - Requirements: CUDA-capable NVIDIA GPU + RAPIDS installation

Note: the CPU baseline above is already fast at this size (285k rows).
GPU acceleration matters at much larger scales - tens of millions of rows.""")

# The groupby was not just a benchmark - read what it found.
print("\nFraud rate by transaction amount (real result of that groupby):")
fraud_by_band = large_data.groupby('amount_band', observed=True)['Class'].agg(
    ['count', 'sum', 'mean'])
fraud_by_band.columns = ['transactions', 'frauds', 'fraud_rate']
print(fraud_by_band.assign(fraud_rate=lambda d: (d['fraud_rate'] * 100).round(3)).to_string())
print("A benchmark that also answers a real question costs nothing extra.")


Benchmark data: 284,807 real transactions x 6 columns
No CUDA GPU here - measuring the pandas (CPU) baseline only

1. Filtering 284,807 rows:
   Pandas (CPU): 0.0007 seconds → 28,837 rows kept

2. Groupby-mean over 5 amount bands:
   Pandas (CPU): 0.0024 seconds

What cuDF would add (NOT measured here - no GPU ran in this notebook):
    - The same filter/groupby API, executed on the GPU
    - NVIDIA's published benchmarks report large speedups on much bigger
      datasets - treat those as the vendor's numbers, not ours
    - Requirements: CUDA-capable NVIDIA GPU + RAPIDS installation

Note: the CPU baseline above is already fast at this size (285k rows).
GPU acceleration matters at much larger scales - tens of millions of rows.

Fraud rate by transaction amount (real result of that groupby):
             transactions  frauds  fraud_rate
amount_band                                  
0-10               100264     249       0.248
10-50               90781      57       0.063
50-200      

## 💬 Discuss

That benchmark also answered a real question. Fraud rate by transaction amount, over
284,807 real transactions: **£0–10: 0.248%**, **£10–50: 0.063%**, **£50–200: 0.156%**,
**£200–1k: 0.293%**, **£1k+: 0.306%**.

1. That is a **U-shape**, not a straight line: fraud is most likely at the very smallest
   and very largest amounts, and least likely in the ordinary middle. Propose an
   explanation for each end of the U. What would you look for in the data to test your
   explanation rather than just asserting it?
2. The 0–10 band has **100,264 transactions and 249 frauds**; the 1k+ band has **2,940
   transactions and 9 frauds**. Both rates are around 0.25–0.31%. Which of those two rates
   do you trust more, and why? Compute roughly how much the small-band rate could move if
   three more frauds were found there.
3. A bank asks you to block transactions in the highest-risk band. Using these numbers,
   estimate how many legitimate customers you block for each fraud you stop. Is that a
   trade you would recommend? What would change your answer?


## Summary

### Key Concepts:
1. **cuDF**: GPU-accelerated DataFrame library with pandas-like API
2. **Import/Export**: Supports CSV, Parquet, JSON, ORC formats
3. **GPU Acceleration**: cuDF runs pandas-style operations on the GPU. On this
   machine's run, if no GPU was present, only the pandas CPU baseline was
   measured — the large speedups quoted for cuDF come from NVIDIA's published
   benchmarks on much larger datasets, not from this notebook
4. **Use Cases**: Large-scale data processing, real-time pipelines, accelerated EDA

### Benefits:
- pandas API compatibility → easy migration when a GPU is available
- Integration with the RAPIDS ecosystem
- Worth reaching for when data is far bigger than the 100k rows used here

**Reference:** Course 05, Unit 2: "Import/Export using cuDF" and "Optimization using cuDF: Using GPU acceleration"


## ⚠️ Where this breaks

- **"Parquet is faster" is a claim about a workload, not a format.** The notebook measured
  3.0× on 6 columns × 284,807 rows with a warm cache, and says outright that re-running
  will move the number. Parquet's advantage grows with the number of columns you *skip*
  and shrinks to nothing when you read every column of a small table.
- **Parquet is not for everything.** It is immutable and columnar: appending a row means
  writing a new file, and reading a single record means touching every column chunk it
  lives in. For row-at-a-time transactional access, use a database. For a file a human
  must open in a text editor or email to a colleague, use CSV.
- **The GPU claims in this notebook are not ours, and it says so.** With no CUDA device
  present, every timing printed here is pandas on the CPU, and the cuDF speedups come from
  NVIDIA's published benchmarks on far larger data. At 284,807 rows the CPU filtered in
  **0.0007 s** — there is nothing here for a GPU to rescue.
- **The assumption that must hold for the fraud analysis: the amount bands are stable over
  time.** These are two days of European card transactions from September 2013. Attack
  patterns move; a rate measured then is a hypothesis now, not a rule.
- **The imbalance will break everything downstream.** 492 frauds in 284,807 rows is 0.173%.
  A model that predicts "not fraud" every time is 99.83% accurate and completely useless —
  which is exactly the trap Unit 4 lesson 06 walks you into deliberately.
- **Cheaper alternative to a GPU, in order:** read fewer columns, convert once to Parquet,
  use categorical dtypes, downcast float64 to float32. Each is free and each attacks the
  bottleneck this notebook actually measured — I/O — rather than the one the marketing
  addresses.


## 📚 References

1. McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in Science Conference (SciPy). <https://doi.org/10.25080/Majora-92bf1922-00a>
2. Raschka, S., Patterson, J., & Nolet, C. (2020). *Machine Learning in Python: Main Developments and Technology Trends in Data Science, Machine Learning, and Artificial Intelligence*. Information, 11(4), 193. <https://arxiv.org/abs/2002.04803>
3. Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). *Array Programming with NumPy*. Nature, 585, 357-362. <https://arxiv.org/abs/2006.10256>
4. Luo, J., Chen, Q., & Binnig, C. (2026). *Do GPUs Really Need New Tabular File Formats?*. arXiv:2602.17335. <https://arxiv.org/abs/2602.17335>
5. Mozzillo, A., Zecchini, L., Gagliardelli, L., Aslam, A., Bergamaschi, S., & Simonini, G. (2025). *Evaluation of Dataframe Libraries for Data Preparation on a Single Machine*. EDBT 2025, 337-349. <https://arxiv.org/abs/2312.11122>
